In [1]:
# fix imports
import os
import sys

module_path = os.path.abspath(os.path.join(".."))
if module_path not in sys.path:
    sys.path.append(module_path)

In [2]:
from src import utils

In [3]:
from huggingface_hub import HfFolder, login

api_file = "/home/fre.gilad/source/llm-iml/HF_KEY.txt"
hf_token = utils.api_key_from_file(api_file)

HfFolder.save_token(hf_token)
login(token=hf_token)

In [4]:
import pandas as pd
from src.data import DF_Batcher

data = pd.read_csv("/home/fre.gilad/source/llm-iml/data/HarmBench/harmful_behaviors.csv")
data = data.rename(columns={"goal": "prompt"})

# shuffle the data
data = data.sample(frac=1).reset_index(drop=True)

In [5]:
# split = int(0.5 * len(data))
# ds_train = data.iloc[:split].copy()
# ds_eval = data.iloc[split:].copy()

ds_train = data.copy()
ds_eval = data.copy()

dl_train = DF_Batcher(ds_train, batch_size=10, shuffle=True)
dl_eval = DF_Batcher(ds_eval, batch_size=50, shuffle=False)

In [6]:
print("Train dataset size:", len(ds_train))
print("Eval dataset size:", len(ds_eval))

Train dataset size: 200
Eval dataset size: 200


In [7]:
from src.eval.harmbench_evaluator import HarmBenchEvaluator
from src.eval.llama_evaluator import LlamaEvaluator
from src.eval.template_evaluator import TemplateEvaluator
from src.eval.llama_guard_evaluator import LlamaGuardEvaluator
from src.eval.strong_reject_evaluator import StrongRejectEvaluator
from gserve.configs import ServeConfig, LLMConfig


evaluators = [
    # HarmBenchEvaluator(
    #     serve_config=ServeConfig(gpu_ids=[1], startup_timeout=10 * 60, client_timeout=60),
    #     use_context=False,
    # ),
    # StrongRejectEvaluator(
    #     serve_config=ServeConfig(gpu_ids=[2], startup_timeout=10 * 60, client_timeout=60),
    #     binary_thresh=0.5,
    # ),
    TemplateEvaluator(),
]

INFO 07-13 20:45:12 [__init__.py:244] Automatically detected platform cuda.


In [8]:
from src.adver_model import AdverModel
import torch
from notebooks.utils import print_supported_models, load_model

torch.set_float32_matmul_precision("high")

print_supported_models()

Supported Models:
- Qwen/Qwen3-0.6B
- Qwen/Qwen2.5-0.5B-Instruct
- GraySwanAI/Llama-3-8B-Instruct-RR
- GraySwanAI/Mistral-7B-Instruct-RR
- Orenguteng/Llama-3-8B-Lexi-Uncensored
- meta-llama/Meta-Llama-3-8B-Instruct
- meta-llama/Llama-3.2-1B-Instruct
- meta-llama/Llama-2-7b-chat-hf
- lmsys/vicuna-7b-v1.5
- mistralai/Mistral-7B-Instruct-v0.3
- tiiuae/falcon-7b-instruct
- tiiuae/Falcon3-7B-Instruct
- mosaicml/mpt-7b-chat
- microsoft/Orca-2-7b
- microsoft/Phi-3-mini-4k-instruct
- microsoft/Phi-4-mini-instruct
- upstage/SOLAR-10.7B-Instruct-v1.0
- openchat/openchat-3.5-0106
- HuggingFaceH4/zephyr-7b-beta
- cais/zephyr_7b_r2d2
- google/gemma-2b-it
- google/gemma-2-2b-it
- google/gemma-3-1b-it
- ContinuousAT/Llama-2-7B-CAT
- apple/OpenELM-1_1B-Instruct


In [9]:
model, tokenizer = load_model("Qwen/Qwen3-0.6B")

Model Config:
model_name: Qwen/Qwen3-0.6B
device_map: cuda:0
torch_dtype: torch.bfloat16
hf_token: None


In [10]:
print(model)

Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 1024)
    (layers): ModuleList(
      (0-27): 28 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=1024, out_features=2048, bias=False)
          (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (o_proj): Linear(in_features=2048, out_features=1024, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (up_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (down_proj): Linear(in_features=3072, out_features=1024, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): Qwen3RMSNorm((1024,), eps=1e-06)
        (post_attention_layernorm): Qwe

In [11]:
# TODO: (low priority) compare float32 + mixed precision preformance vs regular bfloat16 vs float16:
# the loss landscape looks different (more smooth) but the UAP performance when using bfloat16 remains the same

from torch import optim
from src.attacks.soft_prompt import SoftPrompt
from src.iml_attack import IML_Attack
from src.adver_model import AdverModel
from src.initialize import Initializer
from src.activ_extractor import ActivationExtractor
from src.config import GenConfig, StopCriteria


adv_model = AdverModel(model=model, tokenizer=tokenizer, num_tokens=10)

Initializer.normal(adv_model)

internal_attack = SoftPrompt(
    adv_model,
    optim_factory=lambda params: optim.AdamW(params, lr=1e-3),
    steps=15,
    mixed_precision=False,
)

activ_extractor = ActivationExtractor(model, "lm_head", capture_output=False)

gen_config = GenConfig(
    max_length=512,
    do_sample=False,
)

iml_attack = IML_Attack(
    adv_model=adv_model,
    internal_attack=internal_attack,
    optim_factory=lambda params: optim.AdamW(params, lr=2e-2),
    activ_extractor=activ_extractor,
    evaluators=evaluators,
    eval_freq=0.5,
    gen_config=gen_config,
    mixed_precision=False,
    skip_already_fooled=False,
    skip_failed_attacks=True,
    log_dir="logs",
)

stop = StopCriteria(max_epochs=5, max_time=60 * 60)

In [12]:
adv_model = iml_attack.fit(dl_train, dl_eval, stop_criteria=stop)
iml_attack.close()

Logging enabled. Saving logs to: logs/Qwen/Qwen3-0.6B/num_tokens_10/SoftPrompt/2025-07-13_20-45-22
ClearML Task: created new task id=c12eac29d71640c889993c27e532e64d
ClearML results page: https://app.clear.ml/projects/a5f3128511554e9cafdfee319f8a9323/experiments/c12eac29d71640c889993c27e532e64d/output/log


Epochs:   0%|          | 0/5 [00:00<?, ?it/s]

Generating:   0%|          | 0/4 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
# TODO: (low priority) make sure the model computation actually runs at the model dtype
adv_model.set_embeddings(iml_attack.best_embeds)
adv_model.discretize()
iml_attack.evaluate(adv_model=adv_model, dl_eval=dl_eval, evalers=evaluators)

In [ ]:
preds = iml_attack.predict(adv_model, dl_eval, max_length=300)
dl_eval.set_column("response", preds)

for i in range(len(preds)):
    print(f" == Prompt:")
    print(ds_eval.iloc[i]["prompt"])
    print(f" == Target:")
    print(ds_eval.iloc[i]["target"])
    print(f" == Prediction:")
    print(preds[i])
    print("\n" + "=" * 50 + "\n")